# Lost in Transcription — Indonesian ↔ Javanese track

**Objective.** Minimise corpus WER on held-out WhatsApp voice-note dialogues between
Indonesian–Javanese bilinguals, using code that runs unattended in a sealed container:
Python 3.12, one A100 80 GB, 24 vCPU, 220 GB RAM, **no network**, **2 hours wall clock**,
read-only `data/`, and one `submission.csv` at the end.

**The one idea this notebook applies everywhere.** The score is *a normaliser followed by
`jiwer.wer`*. So model exactly what the normaliser keeps, and spend nothing on what it
throws away. Read the scorer first, design the system second. Three consequences drive
every design decision below:

1. Casing is compared **as written** except at a sentence start, so `SMP` vs `smp` is a
   full substitution — a *style* error priced identically to a misheard word.
2. Reduplication hyphens make `jalan-jalan` **one** token; emitting `jalan jalan` costs a
   substitution *and* an insertion. Javanese is full of reduplication.
3. A free-standing `...` is deleted, but an *attached* `wis...` is a real token. The
   references use attached ellipses for disfluencies, which are everywhere in
   spontaneous speech.

None of those three is an acoustic problem, and together they are worth several WER
points (measured in §3). A system that only optimises "did it hear the words" leaves
them on the table.

---

### Solution at a glance

| Layer | Mechanism | Why it wins WER | Code |
|---|---|---|---|
| Metric | Byte-exact re-implementation of `score.py`, plus its fixed point | Local WER == leaderboard WER; targets stop containing tokens the scorer deletes | `lit_jv/scoring.py` |
| Model | Qwen3-ASR (LLM-decoder ASR), context field used as a hotword/style channel | Accepts free-text conditioning at inference, unlike CTC/Whisper-style decoders | `lit_jv/prompts.py`, `lit_jv/decode.py` |
| Adaptation | Two-stage LoRA curriculum + adapter fusion (Javanese lexicon → in-style orthography) | Separates *what words exist* from *how they are written*; fusion weight is a tunable knob | `lit_jv/train_lora.py` |
| Robustness | Length prior + logprob gate → confidence-triggered re-decode ladder | LLM decoders truncate long turns; one lost 35 s turn outweighs 100 casing errors | `lit_jv/decode.py`, `lit_jv/combine.py` |
| Combination | Self-ensemble over prompt variants, merged with ROVER (MBR as fallback) | Extra accuracy from one model, which is all a 2-hour budget affords | `lit_jv/combine.py` |
| Style repair | Orthography rules, each admitted only if it lowers WER on a held-out conversation | Converts the §3 error budget into actual points, with no risk of a plausible-but-harmful rule | `lit_jv/postprocess.py` |
| Delivery | Placeholder submission written before the model loads; per-pass checkpoints; hard deadline | A timeout or a crash scores nothing; this cannot happen | `submission_src/main.py` |

### What is borrowed, and from where

| Source | Idea taken | How it appears here |
|---|---|---|
| Radford et al. 2022, *Robust speech recognition via large-scale weak supervision* | Long-form decoding fails by looping/truncating; gate on avg-logprob and fall back with temperature | `FallbackConfig` (logprob + length-z + finish-reason triggers), `repair_degenerate` |
| Pratap et al. 2024, *Scaling speech technology to 1,000+ languages* (MMS) | Per-language adapter routing | Per-corpus **style tags** in the prompt: one model, routed by a text token instead of a weight swap |
| Omnilingual ASR, 2025 | LLM-decoder ASR conditioned on text context | The `context` field carries a mined Javanese orthography glossary at inference |
| Kulkarni et al. 2023, *Adapting the adapters for code-switching in multilingual ASR* | Train per-regime adapters, then **fuse** them rather than jointly fine-tuning | `fuse_adapters`: interpolate the lexicon adapter and the style adapter, weight chosen on held-out speakers |
| Hillah et al. 2024, *"¿Te vienes? Sure!" Joint fine-tuning for code-switched ASR* | Monolingual + code-switched data jointly beats code-switched only | Stage 1 mixes monolingual Javanese (Jember) with the code-switched dev set; stage 2 specialises |
| Firdaus et al. 2025, *Is Whisper ready for Indonesian?* | Whisper degrades sharply on spontaneous/noisy Indonesian | Rationale for Qwen3-ASR + in-domain LoRA instead of a Whisper baseline |
| Fiscus 1997 (ROVER); Goel & Byrne 2000 (MBR) | Voting/consensus over multiple hypothesis streams | `rover_combine`, `mbr_select` — pure-Python, no extra dependency in the container |

> **Rules compliance.** Every test clip is decoded independently; nothing is fitted on test
> data. All conditioning (glossary, style lexicon, length prior, rule set) is fitted on the
> development set and on MDC training corpora, then frozen into the archive. The container
> needs no network. Nothing about the test set is ever logged.

## 0. How to run this notebook

Sections 1–5, 7–9 and 11–13 are CPU-only and run anywhere in a couple of minutes.
Sections 6 and 10 need the GPU; they are skipped automatically when CUDA is absent and
they cache their outputs under `artifacts/`, so a single GPU session makes the whole
notebook reproducible offline afterwards.

Order of operations for a full run:

1. GPU box, `uv sync` against the competition runtime.
2. Run §1–§5 (parity, EDA, targets, prompts).
3. Run §6 to get zero-shot hypotheses — this is the baseline everything is measured against.
4. Run §10 to fine-tune; re-run §6 pointing at the merged checkpoint.
5. Run §8–§9 to fit post-processing and pick the combiner on held-out conversations.
6. Run §13 to write `submission_config.json`, pack the archive and dry-run the container entrypoint.

When the GPU cells are skipped, §7–§9 fall back to a clearly-labelled **simulated**
hypothesis set (references degraded along the three style dimensions). The machinery is
identical; only the numbers are synthetic, and every table says so.

Everything is controlled by environment variables, so the same notebook runs on a laptop
and on a rented A100 without edits:

| Variable | Default | What it does |
|---|---|---|
| `LIT_SEED` | `3407` | Master seed for data order, augmentation, sampling and init (§0.1). |
| `LIT_DETERMINISTIC` | `0` | `1` asks torch for deterministic kernels — slower, for verifying a result. |
| `LIT_MODEL` | `Qwen/Qwen3-ASR-1.7B` | Decoding checkpoint: a local path, or a Hub repo id. |
| `LIT_HF_REPO` | unset | Hub repo to publish adapters, metrics and manifests to (§10.3). |
| `HF_TOKEN` | unset | Write token. Never read from a file, never logged, never in an artefact. |
| `LIT_HF_UPLOAD_MERGED` | `0` | `1` also pushes the merged 1.7B checkpoint (gigabytes). |
| `LIT_DEV_DIR` | `./indonesian_dev` | Development set. Tracked on a private repo; override only if it lives elsewhere. |
| `LIT_JEMBER` | `./jember` | Extra Javanese corpus for stage 1. |
| `LIT_SKIP_DECODE` / `LIT_SKIP_TRAIN` | `0` | Force the cached/simulated path even with a GPU present. |

TensorBoard event files land in `artifacts/runs/`, and the same directory is what gets
pushed to the Hub, so the loss and held-out-WER curves stay attached to the weights that
produced them.

### 0.0 Runpod setup (run first on a GPU pod)

This cell installs the complete notebook environment — analysis, LoRA training, inference,
tests and packaging — from the repository lock file. Runpod's Debian image can ship a
system-managed `blinker` without pip metadata, so the cell refreshes that one wheel before
installing the rest. It is automatically enabled on Runpod;
on another Linux GPU host, change `INSTALL_RUNPOD_STACK` to `True`. It is skipped on laptops
and CPU-only environments so the analysis-only sections remain usable. Restart the kernel if
this cell upgrades an already-running package.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

NOTEBOOK_ROOT = Path.cwd()
if not (NOTEBOOK_ROOT / 'requirements-runpod.txt').exists():
    NOTEBOOK_ROOT = NOTEBOOK_ROOT.parent

# Auto-enables on the standard Runpod /workspace mount. Set to True for another Linux GPU host.
INSTALL_RUNPOD_STACK = (
    sys.platform == 'linux' and (bool(os.environ.get('RUNPOD_POD_ID')) or Path('/workspace').exists())
)
if INSTALL_RUNPOD_STACK:
    # Installs requirements-runpod.txt, repairing any apt-managed package without a pip RECORD file
    # (blinker, pyparsing, ...), then runs scripts/verify_runpod.py.
    subprocess.run([sys.executable, str(NOTEBOOK_ROOT / 'scripts' / 'install_runpod.py')], check=True)
else:
    print('Runpod setup skipped. Set INSTALL_RUNPOD_STACK = True in this cell for a Linux GPU host.')

In [ ]:
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "lit_jv").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from lit_jv.data import load_dotenv, resolve_dev_dir

_dotenv = load_dotenv(ROOT / ".env")
DEV_DIR = resolve_dev_dir()
SUBMISSION_SRC = ROOT / "submission_src"
ARTIFACTS = ROOT / "artifacts"
HYP_DIR = ARTIFACTS / "hypotheses"
HYP_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 88)

from lit_jv import audio, combine, data, decode, postprocess, prompts, scoring


def show(obj):
    # Renders a DataFrame in Jupyter and prints it when the same code is run as a script.
    if hasattr(obj, "to_string"):
        try:
            from IPython.display import display
        except ImportError:
            print(obj.to_string())
        else:
            display(obj)
    else:
        print(obj)


def installed(name):
    try:
        return importlib.util.find_spec(name) is not None
    except (ImportError, ValueError):
        return False


HAS_TORCH = installed("torch")
HAS_GPU = False
GPU_NAME = ""
if HAS_TORCH:
    import torch

    HAS_GPU = torch.cuda.is_available()
    if HAS_GPU:
        GPU_NAME = torch.cuda.get_device_name(0)

RUN_DECODE = HAS_GPU and os.environ.get("LIT_SKIP_DECODE") != "1"
RUN_TRAIN = HAS_GPU and os.environ.get("LIT_SKIP_TRAIN") != "1"

show(
    pd.DataFrame(
        [
            {"component": "python", "value": sys.version.split()[0]},
            {"component": "torch", "value": (torch.__version__ if HAS_TORCH else "absent")},
            {"component": "cuda device", "value": GPU_NAME or "none"},
            {"component": "vllm / qwen_asr", "value": f"{installed('vllm')} / {installed('qwen_asr')}"},
            {"component": "jiwer / mutagen", "value": f"{installed('jiwer')} / {installed('mutagen')}"},
            {"component": "GPU sections", "value": "ENABLED" if RUN_DECODE else "skipped (cached artifacts / simulation)"},
            {"component": ".env", "value": str(_dotenv) if _dotenv else "not present (copy .env.example -> .env)"},
            {"component": "dev dir", "value": str(DEV_DIR)},
        ]
    ).set_index("component")
)

### 0.1 Reproducibility: seeding, and what a seed can honestly promise

Every number below is produced under a fixed seed, and every artefact is written next to a
`run_manifest.json` that records the seed, a hash of the configuration, a hash of the
training data's *content*, the package versions, the git commit and the GPU model. That
last part is what actually pays off later: when two runs disagree, the diff between their
manifests is the answer, and there is no archaeology to do at 2 a.m. before a deadline.

What a seed does not promise is worth stating plainly, because the opposite is usually
implied. Identical seeds reproduce identical results *on the same software and the same
hardware*. Change the GPU, the CUDA version, the batch size or the number of dataloader
workers and the numbers move slightly, because floating-point reduction order changes.
`LIT_DETERMINISTIC=1` closes most of that gap at a real speed cost, and some fused
attention kernels have no deterministic implementation at all, so it can refuse to run.
Seeds therefore buy comparability between our own runs; the fingerprint buys honesty about
everything else.

Three separate generators have to be pinned, and the third is the one people miss:
Python's `random` (augmentation choices, glossary dropout), NumPy and torch (init,
SpecAugment masks), and each **dataloader worker**, which forks with its own RNG state and
will otherwise silently randomise the augmentation stream across runs.

In [ ]:
from lit_jv import hub, repro

SEED = int(os.environ.get("LIT_SEED", repro.DEFAULT_SEED))
DETERMINISTIC = os.environ.get("LIT_DETERMINISTIC") == "1"
repro.seed_everything(SEED, strict=DETERMINISTIC)

RUNS_DIR = ARTIFACTS / "runs"          # TensorBoard event files live here
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Hub publishing is opt-in and never required: the container has no network, so the
# archive always carries its own weights. The repo is for reference and provenance.
HF_REPO = os.environ.get("LIT_HF_REPO")
HUB = hub.HubSpec.from_env()

FINGERPRINT = repro.environment_fingerprint()
_git = FINGERPRINT["git"]
_torch = FINGERPRINT["torch"] or {}
_versions = {k: v for k, v in FINGERPRINT["packages"].items() if v}

show(
    pd.DataFrame(
        [
            {"field": "seed", "value": SEED},
            {"field": "deterministic kernels", "value": DETERMINISTIC},
            {"field": "commit", "value": (_git.get("commit") or "no git")[:12] + (" +dirty" if _git.get("dirty") else "")},
            {"field": "device", "value": (_torch.get("devices") or ["cpu"])[0]},
            {"field": "cuda / cudnn", "value": f"{_torch.get('cuda')} / {_torch.get('cudnn')}"},
            {"field": "versions", "value": ", ".join(f"{k}={v}" for k, v in sorted(_versions.items()))},
            {"field": "hub repo", "value": HF_REPO or "not configured (LIT_HF_REPO)"},
            {"field": "hub token", "value": "present" if (HUB and HUB.token()) else "absent"},
        ]
    ).set_index("field")
)

if _git.get("dirty"):
    print(f"\nwarning: {_git['dirty_files']} uncommitted file(s) — the recorded commit does "
          "not describe the code that is about to run. Commit before a run you intend to cite.")

## 1. Metric parity — before anything else

`lit_jv/scoring.py` is a verbatim port of the organisers' `normalize_text`, so a WER
computed here is the WER the leaderboard computes. `self_check()` asserts the specific
behaviours the rest of the solution exploits; if it ever fails, every number below is void.

In [ ]:
scoring.self_check()
print("scorer parity assertions passed")

ROWS = data.load_dev_manifest(DEV_DIR, with_durations=True)
print(f"{len(ROWS)} dev clips, {sum(r['duration'] for r in ROWS) / 3600:.2f} h audio")

# Sanity check on the port: reference text is unchanged by the scorer's own normaliser
# once it is at the fixed point, so `text` can be used directly as a training target.
assert all(r["text"] == scoring.to_scoring_space(r["text"]) for r in ROWS)
show(pd.DataFrame(ROWS)[["audio_filename", "speaker", "convo_id", "language", "duration"]].head(5))

### 1.1 The five exploitable properties of the normaliser

Everything the scorer deletes is free to get wrong; everything it keeps is worth points.
The cell below runs the real normaliser, so these are facts, not recollections.

In [ ]:
DEMOS = [
    ("utterance-initial capital is free", "Aku wis mangan"),
    ("initial acronym survives (next letter is also capital)", "SMP-ku adoh"),
    ("mid-utterance casing is compared as written", "aku sekolah ing SMP Negeri 1"),
    ("a bare '...' is deleted outright", "aku ... wis mangan"),
    ("an attached '...' is a scored token", "aku wis... mangan"),
    ("quotes, ; : ! ? cost nothing", 'Halo, "dunia"; ya!'),
    ("em dash collapses to nothing", "aku - wis mangan".replace("-", "\u2014")),
    ("bracketed annotations are deleted", "aku [laugh] wis mangan"),
    ("(?) is deleted, other parens are unwrapped", "aku (?) wis (mangan)"),
    ("a hyphen keeps reduplication as ONE token", "mlaku-mlaku ing pasar"),
    ("... the unhyphenated form is TWO tokens", "mlaku mlaku ing pasar"),
]

show(
    pd.DataFrame(
        [
            {
                "property": label,
                "raw": raw,
                "normalised": scoring.normalize_text(raw),
                "tokens": len(scoring.normalize_text(raw).split()),
            }
            for label, raw in DEMOS
        ]
    ).set_index("property")
)

### 1.2 The normaliser is not idempotent, and it costs a substitution

The sentence-initial rule fires on start-of-string and on `. ! ? —`. A comma is **not** in
that set, but it *is* deleted — and deleted *after* the sentence-initial rule has already
run. So a transcript that begins with a comma keeps its capital through pass one, and a
second pass would lowercase it. The scorer only runs once, which is precisely the problem:
the capital survives into the comparison and mismatches the reference's lowercase word.

That is a real, cheap substitution for any decoder that likes to open its output with
punctuation — and LLM decoders do. Every string entering or leaving this system therefore
goes through `to_scoring_space`, which iterates to the fixed point.

In [ ]:
for probe in (", Aku ora ngerti", "Aku ora ngerti"):
    one = scoring.normalize_text(probe)
    two = scoring.normalize_text(one)
    print(f"{probe!r:22} -> pass1 {one!r:22} -> pass2 {two!r:22}"
          f"  {'STABLE' if one.split() == two.split() else 'UNSTABLE'}")
print("\nfixed point of the leading-comma case:", repr(scoring.to_scoring_space(", Aku ora ngerti")))
print("cost of shipping the pass-1 form against reference 'aku ora ngerti': "
      f"WER {scoring.corpus_wer(['aku ora ngerti'], [scoring.normalize_text(', Aku ora ngerti')]):.3f}")

unstable = [r for r in ROWS if not scoring.is_normalization_fixed_point(scoring.normalize_text(r["raw_text"]))]
print(f"\n{len(unstable)}/{len(ROWS)} dev references are affected, so this is insurance "
      "rather than a live problem on the reference side -")
print("but model output is not drawn from the reference distribution, which is exactly why "
      "the guard sits on the model's side too.")

## 2. The development set, and what it says about the test set

372 clips, 2 conversations, 4 speakers, ~2.5 h. The test set comes from the same corpus
with **no overlapping speaker or conversation**, so the honest local protocol is to hold
out a whole conversation — never a random split. 95% of turns mix both languages; the
remaining 5% are Indonesian only.

In [ ]:
df = pd.DataFrame(ROWS)
df["n_words"] = df["text"].str.split().str.len()
df["wps"] = df["n_words"] / df["duration"]
df["n_chars"] = df["text"].str.len()

show(
    df.groupby(["convo_id", "speaker"])
    .agg(
        clips=("audio_filename", "size"),
        minutes=("duration", lambda s: round(s.sum() / 60, 1)),
        words=("n_words", "sum"),
        median_s=("duration", lambda s: round(s.median(), 1)),
        wps=("wps", lambda s: round(s.mean(), 2)),
    )
    .assign(word_share=lambda t: (t["words"] / t["words"].sum()).round(3))
)
show(df["language"].value_counts().rename("clips").to_frame().T)
print(
    f"duration: min {df.duration.min():.1f}s  median {df.duration.median():.1f}s  "
    f"p95 {df.duration.quantile(0.95):.1f}s  max {df.duration.max():.1f}s"
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12.5, 7.5))

axes[0, 0].hist(df["duration"], bins=40, color="#3b6ea5")
axes[0, 0].set(title="clip duration", xlabel="seconds", ylabel="clips")

axes[0, 1].scatter(df["duration"], df["n_words"], s=12, alpha=0.55, color="#3b6ea5")
prior = combine.LengthPrior.fit(df["text"].tolist(), df["duration"].tolist())
xs = [df["duration"].min(), df["duration"].max()]
for z, style in ((0, "-"), (3, "--"), (-3, "--")):
    axes[0, 1].plot(
        xs, [(prior.mean + z * prior.std) * x for x in xs], style, color="#c0392b", lw=1.2
    )
axes[0, 1].set(title=f"words vs duration (prior {prior.mean:.2f} +/- {prior.std:.2f} w/s, +/-3 sigma)",
               xlabel="seconds", ylabel="reference words")

# Corpus WER pools errors over all reference words, so long clips carry the score.
ordered = df.sort_values("duration", ascending=False)
axes[1, 0].plot(
    range(1, len(ordered) + 1),
    (ordered["n_words"].cumsum() / ordered["n_words"].sum()).values,
    color="#3b6ea5",
)
axes[1, 0].axhline(0.5, ls=":", color="#c0392b")
axes[1, 0].set(title="cumulative share of reference words (longest clips first)",
               xlabel="clips", ylabel="share of scored words")

axes[1, 1].hist(df["wps"], bins=35, color="#3b6ea5")
axes[1, 1].set(title="speaking rate", xlabel="words / second", ylabel="clips")
plt.tight_layout()
plt.show()

n_half = int((ordered["n_words"].cumsum() / ordered["n_words"].sum() <= 0.5).sum()) + 1
print(f"the {n_half} longest clips ({n_half / len(df):.0%} of the set) carry half of all scored words")
print(f"length prior: mean={prior.mean:.4f} std={prior.std:.4f} words/s")
print("+/-3 sigma word budget:",
      {int(t): (round((prior.mean - 3 * prior.std) * t), round((prior.mean + 3 * prior.std) * t))
       for t in (5, 15, 30, 40)})

That cumulative curve is the reason the truncation guard matters more than it looks: a
handful of long turns carry half the scored words, so one silently truncated 35-second
turn costs more than a hundred casing errors. It is also why per-utterance WER is the
wrong thing to average — the metric pools S+D+I over all reference words.

In [ ]:
# A deliberately small, high-precision probe: closed-class function words that belong to
# exactly one of the two languages. This is not language ID and it undercounts heavily
# (shared vocabulary is invisible to it), so read every number below as a LOWER bound.
JV_ONLY = set(
    """ora iso karo wae wis wes durung dhewe piye kepiye mengko saiki kowe isih mung arep
    jare ngono ngene ngunu sing ing nek lek akeh sithik banjur nanging gawe kudu wingi
    sesuk maneh tenan iku iki kuwi kae opo ono ana ojo kabeh mek pisan ambek koyo ket
    wong tak ndhek""".split()
)
ID_ONLY = set(
    """tidak bisa dengan sudah belum untuk sendiri bagaimana nanti sekarang tetapi masih
    hanya saya kamu yang ini itu kalau banyak akan adalah sangat kemudian tapi bukan
    orang harus jalan rumah anak semua boleh jangan seperti waktu""".split()
)
assert not (JV_ONLY & ID_ONLY)


def switch_profile(text):
    toks = [t.lower().strip("-.,") for t in text.split()]
    jv = sum(t in JV_ONLY for t in toks)
    idn = sum(t in ID_ONLY for t in toks)
    return pd.Series(
        {
            "jv_markers": jv,
            "id_markers": idn,
            "jv_ratio": (jv / (jv + idn)) if (jv + idn) else float("nan"),
            "both": jv > 0 and idn > 0,
        }
    )


df = df.join(df["text"].apply(switch_profile))
print(f"{df['both'].mean():.1%} of clips contain unique function words from BOTH languages")
print(f"marker tokens: {int(df['id_markers'].sum())} Indonesian vs {int(df['jv_markers'].sum())} Javanese")
print(f"median Javanese share of marked tokens, per clip: {df['jv_ratio'].median():.3f}")
show(
    df.groupby("language")
    .agg(
        clips=("both", "size"),
        both_languages=("both", "mean"),
        median_jv_share=("jv_ratio", "median"),
        words=("n_words", "sum"),
    )
    .round(3)
)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(df.loc[df["jv_markers"] + df["id_markers"] >= 3, "jv_ratio"].dropna(),
        bins=20, color="#3b6ea5")
ax.set(title="Javanese share of language-marked tokens (clips with >=3 markers)",
       xlabel="jv / (jv + id) markers", ylabel="clips")
plt.tight_layout()
plt.show()

Two things to take from this, both of which shape the design:

**Indonesian is the matrix language and Javanese is embedded in it** — roughly three
Indonesian marker tokens for every Javanese one, and the per-clip Javanese share clusters
low rather than at either extreme. So the base model's Indonesian competence is an asset
worth preserving; the gap is Javanese *lexical coverage and spelling*, which is exactly
what stage 1 of the LoRA curriculum targets and what the glossary biases at inference.

**The mixing is intra-sentential.** Half of all clips contain function words unique to
*both* languages — a strict lower bound, since the probe cannot see shared vocabulary, and
the dataset card reports 95% of turns as mixed. A per-clip language decision, the standard
move in multilingual ASR, is therefore the wrong shape for this data: whatever label you
pick is wrong for part of every clip. So the prompt never asks for a language decision. It
keeps language open and conditions on *style* instead — the transcription conventions of
this corpus (§5).

## 3. The error budget: where WER actually comes from

Before choosing a model, price the failure modes. Each row below degrades the *reference*
along one orthographic dimension and scores it against the untouched reference. The
number is the WER a perfect-hearing but style-blind system pays for that dimension alone —
a floor no amount of acoustic modelling removes.

In [ ]:
ablation = scoring.style_loss_ablation([r["text"] for r in ROWS])
show(
    pd.DataFrame(
        [{"style dimension lost": k, "WER paid": round(v, 4)} for k, v in ablation.items()]
    )
    .sort_values("WER paid", ascending=False)
    .set_index("style dimension lost")
)
print(f"sum of independent style penalties: {sum(ablation.values()):.4f} WER")

In [ ]:
# How much surface is actually at stake: the token inventory these penalties act on.
tokens = [t for r in ROWS for t in r["text"].split()]
total = len(tokens)
inventory = {
    "tokens (scored)": total,
    "hyphenated (reduplication/clitic)": sum("-" in t.strip("-") for t in tokens),
    "attached ellipsis (disfluency)": sum(t.endswith("...") and t != "..." for t in tokens),
    "bare ellipsis (deleted by scorer)": sum(t == "..." for t in tokens),
    "ALL-CAPS acronym (2-5 chars)": sum(t.isupper() and t.isalpha() and 2 <= len(t) <= 5 for t in tokens),
    "capitalised mid-utterance": sum(t[:1].isupper() and not t.isupper() for t in tokens),
    "accented (e-acute etc.)": sum(postprocess.strip_accents(t) != t for t in tokens),
}
show(
    pd.DataFrame(
        [{"token class": k, "count": v, "share": f"{v / total:.2%}"} for k, v in inventory.items()]
    ).set_index("token class")
)
print("\nmost frequent hyphenated forms:",
      ", ".join(w for w, _ in Counter(t for t in tokens if "-" in t.strip("-")).most_common(12)))
print("most frequent acronyms:",
      ", ".join(w for w, _ in Counter(t for t in tokens if t.isupper() and t.isalpha() and 2 <= len(t) <= 5).most_common(12)))

Read the two tables together. Roughly one token in twenty-five is hyphenated, one in fifty
carries a disfluency ellipsis, and casing is live on every capitalised token — and each of
those is priced at a full substitution (hyphens at a substitution *plus* an insertion).
That is why §8 exists as a first-class stage rather than a cosmetic afterthought, and why
the LoRA curriculum has a dedicated **style** stage: the orthographic convention of this
corpus is a learnable target, not noise.

## 4. Targets and mined context

Two artefacts are mined from the development transcripts and frozen into the submission:

* a **glossary** of Javanese orthographic forms — the hotword list handed to the model's
  context field at inference (Omnilingual-ASR style text conditioning), and
* a **style lexicon** — attested casing, accents and hyphenated forms, which is what makes
  the §8 repair rules evidence-based instead of guesswork.

Both are fitted on development data only. The test set shares no speaker and no
conversation with it, so this is training-side conditioning, not leakage: nothing here
looks at test audio or test text, and the same frozen artefacts are applied to every clip.

In [ ]:
dev_texts = [r["text"] for r in ROWS]

# Terms are ranked by "distinctive and stable": frequent here, and *not* explainable as
# ordinary Indonesian, which is what the base model already writes correctly.
GLOSSARY = prompts.mine_glossary(dev_texts, max_terms=96, min_count=2)
EXEMPLARS = prompts.mine_exemplars(dev_texts, n=4)

print(f"{len(GLOSSARY)} glossary terms; first 48:")
print("  " + ", ".join(GLOSSARY[:48]))
print("\nexemplars (style demonstrations for the context field):")
for e in EXEMPLARS:
    print("  -", e)

LEXICON = postprocess.StyleLexicon.fit(dev_texts, min_count=3, dominance=0.9)
print(
    f"\nstyle lexicon: {len(LEXICON.vocab)} types, {len(LEXICON.casing)} casing rules, "
    f"{len(LEXICON.accents)} accent rules, {len(LEXICON.hyphenated)} hyphenated forms"
)
(ARTIFACTS / "glossary.json").write_text(json.dumps(GLOSSARY, ensure_ascii=False, indent=1), "utf-8")

## 5. Prompt design — the cheapest lever in the whole system

Qwen3-ASR takes a free-text `context` field. It is normally sold as a hotword mechanism,
but it is really a **style channel**: the same audio transcribed under different contexts
comes back with different orthographic conventions. Three things go in:

1. **A style tag** (`wa-voicenote-id-jv`, `jember-session-jv`, `read-tts`). Borrowed from
   MMS-style per-language adapter routing, but implemented as a text token so one model
   serves every corpus. Training and inference use the same tag, so the tag at inference
   selects the transcription convention learned from that corpus.
2. **The mined glossary** — Javanese forms the base model would otherwise spell as
   Indonesian.
3. **Prose instructions** describing the code-switching convention: keep each word in the
   language it was spoken in, keep reduplication hyphens, keep disfluency ellipses attached.

`prompts.py` reproduces the model's chat template byte-for-byte, so the container builds
prompts without loading the processor, and the training and inference prompts are
guaranteed identical — the classic silent source of fine-tuning that "does not help".

In [ ]:
PASS_SPECS = [
    (
        "p1-style-glossary",
        prompts.PromptConfig(prose="codeswitch_style", glossary=tuple(GLOSSARY), max_glossary_terms=96),
        0.0,
    ),
    (
        "p2-style-noglossary",
        prompts.PromptConfig(prose="codeswitch_style", glossary=()),
        0.0,
    ),
    (
        "p3-verbatim-freelid",
        prompts.PromptConfig(prose="verbatim", glossary=tuple(GLOSSARY[:32]), force_language=None),
        0.0,
    ),
]

rendered = prompts.build_prompt(PASS_SPECS[0][1], language_field="javind")
print(rendered.replace("<|im_end|>", "<|im_end|>\n"))
print("\n" + "-" * 100)
show(
    pd.DataFrame(
        [
            {
                "pass": name,
                "prose": cfg.prose,
                "glossary terms": len(cfg.glossary),
                "forces language": cfg.force_language or "(free LID)",
                "prompt chars": len(prompts.build_prompt(cfg, language_field="javind")),
            }
            for name, cfg, _ in PASS_SPECS
        ]
    ).set_index("pass")
)

Why the three passes differ deliberately rather than by random seed: ROVER only helps when
the systems make **different** mistakes. Pass 1 is glossary-biased (better Javanese
spelling, small risk of hallucinating a glossary term); pass 2 is unbiased (no glossary
pull); pass 3 drops the forced-language token and asks for verbatim output (different
segmentation and disfluency behaviour). Three cheap, decorrelated views of one model —
which is the only kind of ensemble a two-hour budget can afford.

## 6. Zero-shot decode (GPU)

vLLM is preferred and Transformers is the fallback; `build_backend` tries them in order so
the same notebook runs on a box without vLLM. Hypotheses are cached as JSONL, so §7–§9 are
reproducible on CPU afterwards.

`max_tokens` is set **per clip** from duration and the fitted length prior, rather than one
global cap. A global cap either truncates the long turns that carry half the score, or
wastes the budget padding short ones. The WER probe during training uses the same generation
path, including the Runpod Qwen return-shape guard, so a missing WER curve is treated as a
failed validation run rather than evidence of a good adapter.

In [ ]:
MODEL_PATH = os.environ.get("LIT_MODEL", "Qwen/Qwen3-ASR-1.7B")
MERGED_DIR = ARTIFACTS / "merged-qwen3asr"
MODEL_CACHE = ARTIFACTS / "models" / str(MODEL_PATH).replace("/", "--")
if MERGED_DIR.exists():
    MODEL_PATH = str(MERGED_DIR)  # after §10 has been run
elif os.environ.get("LIT_HF_CHECKPOINT") and RUN_DECODE:
    # A finished checkpoint on the Hub, pulled once into the local cache. Doing this
    # here rather than inside the container is the whole point: the container has no
    # network, so the download must happen while packaging.
    MODEL_PATH = str(
        hub.resolve_checkpoint(
            os.environ["LIT_HF_CHECKPOINT"],
            subfolder=os.environ.get("LIT_HF_SUBFOLDER", "merged"),
            revision=os.environ.get("LIT_HF_REVISION"),
        )
    )
elif RUN_DECODE and not Path(MODEL_PATH).is_dir():
    # Download once, before backend construction. This turns a vague vLLM model
    # loading failure into a clear Hub/network error and leaves a reusable local copy.
    if os.environ.get("HF_HUB_ENABLE_HF_TRANSFER") and not installed("hf_transfer"):
        os.environ.pop("HF_HUB_ENABLE_HF_TRANSFER", None)
        print("hf_transfer is unavailable; using the standard Hugging Face downloader")
    try:
        MODEL_PATH = str(hub.resolve_checkpoint(MODEL_PATH, local_dir=MODEL_CACHE))
    except Exception as exc:
        raise RuntimeError(
            f"Could not download the base model {MODEL_PATH!r}. Check internet access, "
            "HF_TOKEN (if required), and the Runpod setup cell."
        ) from exc
print("model:", MODEL_PATH)


def hypotheses_path(name, tag="zeroshot"):
    return HYP_DIR / f"{tag}.{name}.jsonl"


def decode_or_load(rows, tag="zeroshot", specs=PASS_SPECS, backend=None):
    # Returns {pass_name: [text, ...]} aligned with `rows`. Cached passes are never re-run.
    out, missing = {}, []
    for name, cfg, temperature in specs:
        path = hypotheses_path(name, tag)
        if path.exists():
            cached = {r["audio_filename"]: r for r in data.read_jsonl(path)}
            out[name] = [cached[r["audio_filename"]] for r in rows]
        else:
            missing.append((name, cfg, temperature))
    if not missing:
        return out
    if not RUN_DECODE:
        print(f"no GPU: {len(missing)} pass(es) not decoded ({[m[0] for m in missing]})")
        return out

    backend = backend or decode.build_backend(MODEL_PATH, prefer="vllm")
    decoder = decode.Qwen3AsrDecoder(backend=backend, length_prior=prior)
    for name, cfg, temperature in missing:
        started = time.time()
        config = decode.DecodeConfig(prompt=cfg, temperature=temperature, name=name)
        hyps = decoder.decode_pass(rows, config, chunk_size=48)
        hyps = decoder.rescue(rows, hyps, config)
        records = [
            {
                "audio_filename": h.audio_filename,
                "text": h.text,
                "avg_logprob": h.avg_logprob,
                "finish_reason": h.finish_reason,
                "n_tokens": h.n_tokens,
                "rescued_by": h.rescued_by,
            }
            for h in hyps
        ]
        data.write_jsonl(records, hypotheses_path(name, tag))
        elapsed = time.time() - started
        print(f"{name}: {len(rows)} clips in {elapsed:.0f}s ({len(rows) / elapsed:.2f} clips/s)")
        out[name] = records
    return out


ZERO_SHOT = decode_or_load(ROWS, tag="zeroshot")

In [ ]:
REFS = [r["text"] for r in ROWS]
DURATIONS = [r["duration"] for r in ROWS]
RESULTS = []  # the ablation ledger, filled in as the notebook proceeds


def record(name, hyps, note=""):
    report = scoring.wer_report(REFS, hyps)
    row = report.as_row(name)
    row["note"] = note
    RESULTS.append(row)
    return row


if ZERO_SHOT:
    for name, records in ZERO_SHOT.items():
        record(name, [r["text"] for r in records], "zero-shot, raw model output")
    show(pd.DataFrame(RESULTS).set_index("system"))
    flagged = {
        name: sum(1 for r in records if r.get("rescued_by")) for name, records in ZERO_SHOT.items()
    }
    print("clips altered by the rescue ladder:", flagged)
else:
    print("no hypotheses yet - run 6 on a GPU box, or continue with the simulation below")

### 6.1 Working without a GPU: a labelled simulation

Sections 7–9 need *some* hypothesis set. When the real ones are absent, the cell below
manufactures them by degrading the references along exactly the failure modes measured in
§3 — style loss on all three dimensions, plus a little lexical noise, with a different
seed per pass so the passes disagree the way real passes do. Every table downstream is
tagged `SIMULATED` when this path is taken. The code exercised is identical; only the
inputs are synthetic.

In [ ]:
SIMULATED = not ZERO_SHOT


def _degrade(text, rng, lexical=0.06):
    words = []
    for w in text.split():
        if w.endswith("...") and w != "...":
            w = w[:-3] + " ..."             # detach the disfluency ellipsis
        if "-" in w.strip("-") and rng.random() < 0.8:
            w = w.replace("-", " ")         # split reduplication
        if rng.random() < 0.5:
            w = w.lower()                   # lose casing
        if rng.random() < lexical:
            continue                        # deletion
        words.append(w)
        if rng.random() < lexical / 3:
            words.append(rng.choice(["ya", "eh", "iku"]))  # insertion
    return " ".join(words)


if SIMULATED:
    PASSES = {}
    for i, (name, _, _) in enumerate(PASS_SPECS):
        rng = random.Random(1000 + i)
        PASSES[name] = [_degrade(t, rng, lexical=0.05 + 0.015 * i) for t in REFS]
    for name, texts in PASSES.items():
        record(f"{name} [SIMULATED]", texts, "synthetic: references degraded per 3")
else:
    PASSES = {name: [r["text"] for r in records] for name, records in ZERO_SHOT.items()}

BEST_PASS = min(PASSES, key=lambda k: scoring.corpus_wer(REFS, PASSES[k]))
print(("SIMULATED " if SIMULATED else "") + f"best single pass: {BEST_PASS} "
      f"WER={scoring.corpus_wer(REFS, PASSES[BEST_PASS]):.4f}")
show(pd.DataFrame(RESULTS).set_index("system"))

## 7. Error taxonomy: what would a repair rule actually be repairing?

`token_error_pairs` walks the jiwer alignment and returns every `(reference,
hypothesis)` mismatch. Classifying those pairs tells us how much of the WER is *style* —
recoverable by a deterministic rule that never touches the audio — and how much is
genuinely misheard. The split decides where the next hour of work should go.

In [ ]:
def classify(ref_token, hyp_token):
    if not ref_token:
        return "insertion: bare ellipsis" if hyp_token == "..." else "insertion: other"
    if not hyp_token:
        return "deletion: bare ellipsis" if ref_token == "..." else "deletion: other"
    if ref_token == hyp_token:
        return "identical (alignment artefact)"
    if ref_token.lower() == hyp_token.lower():
        return "casing only"
    strip = postprocess.strip_accents
    if strip(ref_token).lower() == strip(hyp_token).lower():
        return "accent only"
    if ref_token.replace("-", "") == hyp_token.replace("-", "") or (
        ref_token.replace("-", " ") == hyp_token
    ):
        return "hyphen / reduplication"
    if ref_token.rstrip(".") == hyp_token.rstrip(".") and (
        ref_token.endswith("...") or hyp_token.endswith("...")
    ):
        return "ellipsis attachment"
    if strip(ref_token.replace("-", "")).lower() == strip(hyp_token.replace("-", "")).lower():
        return "mixed style (casing+accent+hyphen)"
    return "lexical / acoustic"


pairs = scoring.token_error_pairs(REFS, PASSES[BEST_PASS])
counts = Counter(classify(a, b) for a, b in pairs)
tax = (
    pd.DataFrame([{"error class": k, "tokens": v} for k, v in counts.items()])
    .sort_values("tokens", ascending=False)
    .assign(share=lambda t: (t["tokens"] / t["tokens"].sum()).map("{:.1%}".format))
    .set_index("error class")
)
show(tax)
style_share = sum(v for k, v in counts.items() if k not in ("lexical / acoustic", "insertion: other", "deletion: other", "identical (alignment artefact)")) / max(1, sum(counts.values()))
print(("SIMULATED. " if SIMULATED else "") + f"style-recoverable share of all token errors: {style_share:.1%}")
print("\nexamples of the largest recoverable classes:")
seen = set()
for a, b in pairs:
    label = classify(a, b)
    if label in ("casing only", "hyphen / reduplication", "ellipsis attachment", "accent only") and label not in seen:
        seen.add(label)
        print(f"  {label:26s} reference={a!r:28s} hypothesis={b!r}")

## 8. Post-processing that has to earn its place

Orthography repair is where style-recoverable errors get converted into points — and also
where a confident-sounding rule can quietly *raise* WER. Two guards:

1. **Forward selection against WER.** `RulePipeline.fit` adds a rule only if the whole
   pipeline, re-run from scratch, scores better with it. Rule interactions are therefore
   priced, not assumed.
2. **Held-out conversations.** Rules are fitted on one conversation and scored on the
   other — the local analogue of the real split, since the test set shares no speaker or
   conversation with dev. The *held-out* delta is the number worth believing.

Only unicode folding is mandatory: a U+2026 ellipsis or a curly apostrophe is never right,
because the references contain neither and the scorer does not fold them.

In [ ]:
convos = sorted({r["convo_id"] for r in ROWS})
fold_rows = []
for held in convos:
    train_idx, held_idx = data.holdout_by_conversation(ROWS, held)
    train_refs = [REFS[i] for i in train_idx]
    train_hyps = [PASSES[BEST_PASS][i] for i in train_idx]
    held_refs = [REFS[i] for i in held_idx]
    held_hyps = [PASSES[BEST_PASS][i] for i in held_idx]
    held_durations = [DURATIONS[i] for i in held_idx]

    # The lexicon may only see the training conversation, or the "held-out" delta is a lie.
    fold_lexicon = postprocess.StyleLexicon.fit(train_refs, min_count=3)
    fitted = postprocess.RulePipeline(lexicon=fold_lexicon).fit(
        train_refs, train_hyps, verbose=False
    )
    before = scoring.corpus_wer(held_refs, held_hyps)
    after = scoring.corpus_wer(held_refs, fitted.apply_many(held_hyps, held_durations))
    fold_rows.append(
        {
            "held-out convo": held,
            "clips": len(held_idx),
            "WER before": round(before, 4),
            "WER after": round(after, 4),
            "delta": round(after - before, 4),
            "rules kept": len(fitted.rules),
        }
    )
show(pd.DataFrame(fold_rows).set_index("held-out convo"))
print(("SIMULATED. " if SIMULATED else "") +
      "a negative delta on a conversation the rules never saw is the only evidence that counts")

In [ ]:
# Final pipeline: fitted on the whole development set, saved next to main.py.
PIPELINE = postprocess.RulePipeline(lexicon=LEXICON).fit(
    REFS, PASSES[BEST_PASS], verbose=True
)
show(
    pd.DataFrame(
        [{"rule": k, "WER gain when added": round(-v, 5)} for k, v in PIPELINE.deltas.items()]
    ).set_index("rule")
)
record(
    f"{BEST_PASS} + post-processing" + (" [SIMULATED]" if SIMULATED else ""),
    PIPELINE.apply_many(PASSES[BEST_PASS], DURATIONS),
    "rules gated on held-out WER",
)
PIPELINE.save(SUBMISSION_SRC / "postprocess.json")
print(f"\nsaved {len(PIPELINE.rules)} rules + lexicon to {SUBMISSION_SRC / 'postprocess.json'}")
show(pd.DataFrame(RESULTS).set_index("system"))

## 9. System combination: ROVER, MBR, and the ceiling above both

Two combiners, both implemented from scratch so the container needs no extra dependency:

* **ROVER** (Fiscus 1997) builds a confusion network from the pass hypotheses and votes
  slot by slot. It can produce a transcript better than *every* input, because it fixes
  each error independently.
* **MBR** (Goel & Byrne 2000) picks the single hypothesis with the lowest expected edit
  distance to the others. It cannot beat its best input, but it never produces the
  word-salad that a bad ROVER vote can.

The oracle row is per-clip best-of-N chosen *with* the references. It is the ceiling for
any **selection** scheme, including MBR — but not for ROVER, which can go past it by
taking the right word from a different pass at each slot. When ROVER beats the oracle, the
passes are disagreeing in a genuinely complementary way and another pass is likely worth
its minutes; when ROVER sits between MBR and the oracle, the passes are too correlated and
the budget is better spent elsewhere. Either way this is a measurement, not a guess.

In [ ]:
names = list(PASSES)
combo_rows = []
rover_texts, mbr_texts, oracle_texts, agreements = [], [], [], []
for i in range(len(ROWS)):
    candidates = [PASSES[n][i] for n in names]
    result = combine.rover_combine(candidates)
    rover_texts.append(result.text)
    agreements.append(result.agreement)
    mbr_texts.append(combine.mbr_select(candidates).text)
    oracle_texts.append(min(candidates, key=lambda c: scoring.corpus_wer([REFS[i]], [c])))

for label, texts in (
    ("best single pass", PASSES[BEST_PASS]),
    ("MBR select", mbr_texts),
    ("ROVER vote", rover_texts),
    ("ROVER + post-processing", PIPELINE.apply_many(rover_texts, DURATIONS)),
    ("oracle best-of-N (ceiling)", oracle_texts),
):
    combo_rows.append(
        {
            "system": label + (" [SIMULATED]" if SIMULATED else ""),
            "WER": round(scoring.corpus_wer(REFS, texts), 4),
        }
    )
show(pd.DataFrame(combo_rows).set_index("system"))
best, mbr_wer, rover_wer, oracle_wer = (combo_rows[i]["WER"] for i in (0, 1, 2, 4))
print(f"mean ROVER slot agreement: {sum(agreements) / len(agreements):.3f}")
print(f"selection headroom (best single -> oracle): {best - oracle_wer:.4f} WER")
print(
    "ROVER is past the selection ceiling: the passes are complementary, another pass is "
    "probably worth its minutes"
    if rover_wer < oracle_wer
    else "ROVER is inside the selection ceiling: the passes are correlated, spend the "
    "budget on the model instead"
)

record("ROVER + post-processing" + (" [SIMULATED]" if SIMULATED else ""),
       PIPELINE.apply_many(rover_texts, DURATIONS), "final decoding stack")

### 9.1 The combiner depends on how many passes survive the clock

This is the trap. `main.py` drops a pass when the clock says so, and **neither combiner has
a real vote at two hypotheses**. Every ROVER slot where the passes disagree is a 1–1 tie,
resolved by the tie-break rule rather than by evidence. MBR is no better off: with two
candidates the two risks differ only through the reference-length normalisation in the
WER loss, so its choice is an artefact of which hypothesis is longer.

Both are therefore *data-dependent* at N=2 rather than reliably good or reliably bad, which
means the candidate set has to include **not combining at all** — just ship the strongest
pass. The table below measures all three at every surviving pass count and writes the
result to the config as `combiner_by_n`; `main.py` reads it with the count that actually
finished. Re-run this cell after fine-tuning: a result that rests on tie-breaking is
exactly the kind that does not transfer between models.

In [ ]:
matrix = []
COMBINER_BY_N = {}
for n in range(2, len(names) + 1):
    subset_names = names[:n]
    scores = {}
    for label in ("mbr", "rover"):
        texts = []
        for i in range(len(ROWS)):
            candidates = [PASSES[m][i] for m in subset_names]
            merged = (
                combine.rover_combine(candidates).text
                if label == "rover"
                else combine.mbr_select(candidates).text
            )
            texts.append(merged)
        scores[label] = scoring.corpus_wer(REFS, PIPELINE.apply_many(texts, DURATIONS))
    single = scoring.corpus_wer(REFS, PIPELINE.apply_many(PASSES[subset_names[0]], DURATIONS))
    winner = min(scores, key=scores.get)
    COMBINER_BY_N[n] = winner if scores[winner] < single else "none"
    matrix.append(
        {
            "passes": n,
            "pass 1 only": round(single, 4),
            "MBR": round(scores["mbr"], 4),
            "ROVER": round(scores["rover"], 4),
            "selected": COMBINER_BY_N[n],
        }
    )
show(pd.DataFrame(matrix).set_index("passes"))
COMBINER = COMBINER_BY_N[len(names)]
print(("SIMULATED. " if SIMULATED else "") + f"combiner_by_n = {COMBINER_BY_N}")
if COMBINER_BY_N.get(2) == "rover":
    print("note: 2-way ROVER wins here on tie-breaking alone - the least transferable "
          "result in this notebook, so re-measure it on real hypotheses")

### 9.2 Guarding against the failure mode that actually loses competitions

Everything above trades in fractions of a point. Truncation trades in whole points: an
LLM decoder that emits EOS halfway through a 35-second turn turns every remaining word
into a deletion. Whisper's own long-form heuristics inspired the ladder used here —
detect, then re-decode differently rather than accept:

1. **Detect** — average logprob below threshold, words-per-second more than 3σ from the
   fitted prior, an empty output, or `finish_reason == "length"`.
2. **Re-decode** — free-LID prompt, then temperature sampling with MBR over the draws,
   then split the clip at its quietest interior point and stitch.
3. **Cap the blast radius** — at most 15% of clips may be rescued, so a mis-calibrated
   threshold cannot consume the whole time budget.

In [ ]:
prior_words = lambda d, z: max(0, round((prior.mean + z * prior.std) * d))
show(
    pd.DataFrame(
        [
            {
                "duration": f"{d:.0f}s",
                "plausible words (-3 sigma .. +3 sigma)": f"{prior_words(d, -3)} .. {prior_words(d, 3)}",
                "flag at": f"<{prior_words(d, -3)} or >{prior_words(d, 3)} words",
            }
            for d in (5, 10, 20, 30, 40)
        ]
    ).set_index("duration")
)

# What the guard would have caught on the current hypothesis set.
fb = decode.FallbackConfig()
detector = decode.Qwen3AsrDecoder(backend=None, length_prior=prior, fallback=fb, log=lambda m: None)
flags = []
for i, row in enumerate(ROWS):
    hypothesis = decode.Hypothesis(
        audio_filename=row["audio_filename"], text=PASSES[BEST_PASS][i], avg_logprob=float("nan")
    )
    reason = detector.is_suspicious(hypothesis, row["duration"])
    if reason:
        flags.append({"reason": reason.split("=")[0], "duration": round(row["duration"], 1)})
print(f"{len(flags)}/{len(ROWS)} clips flagged by the length/finish guards alone "
      f"({len(flags) / len(ROWS):.1%}; cap is {fb.max_rescue_fraction:.0%})")
if flags:
    show(pd.DataFrame(flags).groupby("reason").agg(clips=("duration", "size"),
                                                   median_s=("duration", "median")))

# Degenerate-repair demo. The collapse fires only when the implied speaking rate is
# physically impossible, because the references genuinely contain repetition
# ("tolong tolong tolong tolong") and unconditional collapsing would corrupt it.
loop = ("aku wis mangan " * 20).strip()
genuine = "tolong tolong tolong tolong"
show(
    pd.DataFrame(
        [
            {
                "output": label,
                "clip": f"{seconds:.0f}s",
                "words/s": round(postprocess.words_per_second(text, seconds), 1),
                "flagged": postprocess.looks_degenerate(text, seconds),
                "words in -> out": f"{len(text.split())} -> "
                f"{len(postprocess.repair_degenerate(text, seconds).split())}",
            }
            for label, text, seconds in (
                ("looping decoder", loop, 10.0),
                ("looping decoder", loop, 40.0),
                ("genuine repetition", genuine, 8.0),
            )
        ]
    ).set_index("output")
)

## 10. LoRA adaptation: a curriculum, not a fine-tune (GPU)

Zero-shot Qwen3-ASR knows Indonesian well and Javanese much less well, and it has never
seen this corpus's transcription conventions. Those are two different problems and they
want different data, so they get different stages — the joint-fine-tuning result of
Hillah et al. (2024) says mixing monolingual and code-switched data beats code-switched
data alone, and the adapter-fusion result of Kulkarni et al. (2023) says the two skills
compose better as separate adapters than as one blended objective.

| Stage | Data | Learning rate | Goal |
|---|---|---|---|
| `s1-lexicon` | Jember Javanese sessions (~10 h) + MDC Javanese + dev, 2 epochs | 1.2e-4 | Javanese vocabulary, phonotactics, spontaneous acoustics |
| `s2-style` | dev-heavy (Jember at 0.15 weight), 4 epochs | 3.5e-5 | This corpus's orthography: hyphens, ellipses, casing, accents |

Implementation details that matter more than the hyperparameters:

* **LoRA on both towers.** Audio encoder *and* text decoder. Text-only adapters cannot fix
  Javanese phonotactics; audio-only adapters cannot fix orthography.
* **Targets in scoring space.** Supervising raw transcripts teaches the model to emit
  tokens the scorer deletes, wasting capacity and inviting truncation.
* **Prompt parity.** `training_prompt` builds the exact string `decode.py` builds, with
  glossary dropout at 0.3 so the model learns to *use* the context field without becoming
  dependent on it.
* **Augmentation on out-of-domain corpora only** (speed 0.9/1.0/1.1, gain jitter,
  SpecAugment). Jember is 44.1 kHz session audio, the test set is 48 kHz WhatsApp voice
  notes; augmentation narrows that channel gap. The in-domain dev audio is left alone.
* **Boundary snapping.** Jember timestamps are 1-second resolution, which clips word
  onsets. Segments are padded and then snapped to the quietest point nearby, so targets
  are not systematically truncated relative to the audio.

In [ ]:
JEMBER_DIR = Path(os.environ.get("LIT_JEMBER", ROOT / "jember"))
SEGMENTS_DIR = ARTIFACTS / "jember-segments"

train_parts = [ROWS]
if (SEGMENTS_DIR / "manifest.jsonl").exists():
    jember_rows = data.read_jsonl(SEGMENTS_DIR / "manifest.jsonl")
    train_parts.append(jember_rows)
    print(f"reusing {len(jember_rows)} cached Jember segments")
elif JEMBER_DIR.exists():
    tsv = next(iter(sorted(JEMBER_DIR.rglob("*.tsv"))), None)
    audio_dir = next((p for p in (JEMBER_DIR / "audio", JEMBER_DIR) if p.exists()), JEMBER_DIR)
    if tsv is not None:
        jember_rows = data.segment_session_corpus(
            audio_dir=audio_dir,
            tsv_path=tsv,
            out_dir=SEGMENTS_DIR,
            config=data.SegmentationConfig(),
        )
        data.write_jsonl(jember_rows, SEGMENTS_DIR / "manifest.jsonl")
        train_parts.append(jember_rows)
else:
    print(f"Jember corpus not found at {JEMBER_DIR} - stage 1 will run on dev data only.\n"
          "Download it from the MDC catalog and set LIT_JEMBER to its directory.")

NAHUATL_DIR = Path(os.environ.get("LIT_NAHUATL_DIR", ROOT / "nahuatl_dev"))
if NAHUATL_DIR.exists() and any(NAHUATL_DIR.rglob("*.mp3")):
    nahuatl_rows = data.load_extracted_mdc_corpus(
        NAHUATL_DIR, corpus="nahuatl", language="nah"
    )
    train_parts.append(nahuatl_rows)
    print(f"loaded {len(nahuatl_rows)} Tetelancingo Nahuatl clips from {NAHUATL_DIR}")
else:
    print("no Nahuatl extra corpus (optional). Download with:\n"
          "  python scripts/download_mdc_dataset.py   # needs MDC_API_KEY in .env")

TRAIN_ROWS = data.build_training_manifest(train_parts, use_scoring_space_targets=True)

# A held-out conversation, not a random 5%. Training on part of a conversation and
# validating on the rest shares speaker, channel and topic, and the resulting curve
# improves while the leaderboard does not. The eval split is what the in-training WER
# probe decodes, so it has to be honest or the curve is decorative.
#
# The dev set contains exactly two conversations, at 21% and 79% of the clips, so the
# choice is which one to give up: hold out the larger and 79% of the in-style data is
# gone from training, which is the scarce resource. Pick the conversation closest to a
# quarter of the corpus.
_convo_sizes = Counter(r["convo_id"] for r in ROWS)
EVAL_CONVO = min(_convo_sizes, key=lambda c: abs(_convo_sizes[c] / len(ROWS) - 0.25))
_fit_idx, _eval_idx = data.holdout_by_conversation(TRAIN_ROWS, EVAL_CONVO)
FIT_ROWS = data.subset(TRAIN_ROWS, _fit_idx)
EVAL_ROWS = data.subset(TRAIN_ROWS, _eval_idx)

show(pd.DataFrame(data.manifest_summary(TRAIN_ROWS)).set_index("corpus"))
print(f"conversations: {dict(_convo_sizes)}")
print(f"held out {EVAL_CONVO}: {len(EVAL_ROWS)} clips for eval, {len(FIT_ROWS)} for fitting")
print("data fingerprint:", repro.manifest_fingerprint(FIT_ROWS)["sha256_16"])
data.write_jsonl(TRAIN_ROWS, ARTIFACTS / "train_manifest.jsonl")

In [ ]:
from lit_jv import train_lora
from lit_jv.train_lora import LoraSpec, TrainSpec

SPEC = TrainSpec(
    model_path=os.environ.get("LIT_BASE_MODEL", "Qwen/Qwen3-ASR-1.7B"),
    output_dir=str(ARTIFACTS / "qwen3asr-lora"),
    # Only 2.5 h of target data is present in this run.  Keep the acoustic tower
    # frozen: the previous 49M-parameter adaptation overfit it and hurt the held-out
    # conversation.  A compact decoder-only adapter is the controlled baseline.
    lora=LoraSpec(r=16, alpha=32, dropout=0.05, adapt_text_decoder=True, adapt_audio_encoder=False),
    prompt=prompts.PromptConfig(glossary=tuple(GLOSSARY)),
    train_with_glossary=True,
    glossary_dropout=0.3,
    seed=SEED,
    deterministic=DETERMINISTIC,
    # This run has roughly 20--60 optimiser steps per stage, so a 200-step WER
    # interval never fires.  WER is the checkpoint-selection metric, not loss.
    tensorboard_dir=str(RUNS_DIR),
    report_to=("tensorboard",),
    run_name=f"qwen3asr-lora-seed{SEED}",
    # Trainer's token-level eval expects pre-tokenised input_ids, while this
    # audio manifest is collated lazily.  WER below is the real selection
    # signal, so skip proxy eval loss rather than constructing its sampler.
    eval_steps=0,
    wer_eval_steps=10,
    wer_eval_clips=len(EVAL_ROWS),
    hub=HUB,
)
S1, S2 = SPEC.stages

show(
    pd.DataFrame(
        [
            {
                "stage": s.name,
                "corpus weights": json.dumps(s.corpus_weights),
                "epochs": s.epochs,
                "lr": s.learning_rate,
                "effective batch": s.batch_size * s.grad_accumulation,
                "expanded rows": len(train_lora.expand_stage_rows(FIT_ROWS, s, seed=SEED)),
            }
            for s in (S1, S2)
        ]
    ).set_index("stage")
)
print("config fingerprint:", repro.config_fingerprint(SPEC))

In [ ]:
ADAPTERS = {}
HISTORY = {}
if RUN_TRAIN:
    # The measured fusion sweep selected the pure style adapter (weight 1.0); every
    # mixture was markedly worse.  Train that one evidence-backed curriculum rather
    # than spending GPU time on a harmful fusion candidate.
    for name, stages in (("style", [S1, S2]),):
        result = train_lora.train(
            FIT_ROWS, SPEC, stages=stages, save_adapter_as=f"adapter-{name}", eval_rows=EVAL_ROWS
        )
        ADAPTERS[name] = result["adapter_dir"]
        HISTORY[name] = result
        print(f"{name}: best held-out WER {(result['best_wer'] or {}).get('eval_wer', 'n/a')}")
else:
    for name in ("adapter-lexicon", "adapter-style"):
        candidate = Path(SPEC.output_dir) / name
        if (candidate / "adapter_config.json").exists():
            ADAPTERS[name.split("-")[1]] = str(candidate)
    print("training skipped;", f"found adapters: {list(ADAPTERS)}" if ADAPTERS else "no cached adapters")

### 10.1 Watching the right curve

Training loss goes down. It is the least informative curve in the notebook, and reading it
as progress is how a fine-tune ends up worse than the checkpoint it started from.

Held-out **eval loss** is better, but still a proxy, and a biased one here: it rewards
reproducing punctuation and casing in exactly the places the scorer deletes them, and it
barely notices the one failure that costs the most — a truncated or looping hypothesis,
which is a handful of tokens of loss and a whole clip's worth of WER.

So the probe that matters decodes 48 held-out clips with greedy search and scores them with
the competition's own normaliser, every 200 steps. That is expensive inside a training loop
and worth every second of it: the two curves routinely disagree, and when they do,
`eval/wer` is the one telling the truth. Watch for the classic style-stage signature — eval
loss flat or drifting up while WER improves, because the model is dropping punctuation the
scorer would have deleted anyway.

Everything lands in `artifacts/runs/<stage>/`: `train/loss`, `eval/loss`, `eval/wer` and
`eval/empty_outputs`. Launch TensorBoard beside the notebook,

```bash
tensorboard --logdir artifacts/runs --port 6006
```

or inline, with `%load_ext tensorboard` and `%tensorboard --logdir artifacts/runs`. The
cell below plots the same data from the saved history, so the curves survive a kernel
restart and can be read on a laptop long after the GPU box is gone.

In [ ]:
def load_history(adapter_dir):
    path = Path(adapter_dir) / "train_history.json"
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else []


curves = {name: load_history(path) for name, path in ADAPTERS.items()}
have_curves = any(curves.values())

if have_curves:
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
    for name, history in curves.items():
        for stage in history:
            steps = [e["step"] for e in stage["log_history"] if "loss" in e]
            loss = [e["loss"] for e in stage["log_history"] if "loss" in e]
            if steps:
                axes[0].plot(steps, loss, lw=1.1, label=f"{name}/{stage['stage']} train")
            eval_steps = [e["step"] for e in stage["log_history"] if "eval_loss" in e]
            eval_loss = [e["eval_loss"] for e in stage["log_history"] if "eval_loss" in e]
            if eval_steps:
                axes[0].plot(eval_steps, eval_loss, lw=1.4, ls="--", label=f"{name}/{stage['stage']} eval")
            probe = stage.get("wer_history") or []
            if probe:
                axes[1].plot([e["step"] for e in probe], [e["eval_wer"] for e in probe],
                             marker="o", ms=3, lw=1.3, label=f"{name}/{stage['stage']}")
    axes[0].set_title("loss")
    axes[1].set_title(f"held-out WER (conversation {EVAL_CONVO}, competition normaliser)")
    for ax in axes:
        ax.set_xlabel("optimiser step")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    best = [
        {"adapter": name, "stage": stage["stage"], **(stage.get("best_wer") or {})}
        for name, history in curves.items()
        for stage in history
        if stage.get("best_wer")
    ]
    if best:
        show(pd.DataFrame(best).set_index(["adapter", "stage"]))
else:
    print("no training history yet. After a GPU run, artifacts/qwen3asr-lora/adapter-*/\n"
          "train_history.json holds the loss and WER-probe curves, and artifacts/runs/ holds\n"
          "the TensorBoard event files for the same run.")

### 10.2 Choosing the fusion weight, then merging

The fusion weight `alpha` interpolates the two adapters. It is picked by decoding a
**held-out conversation** at a few values and taking the best WER — never by looking at
the training conversation, where more style specialisation always looks better.

One caveat, stated rather than hidden: with two conversations in the corpus there is no
third split, so the same held-out conversation both selects `alpha` and reports its WER.
That makes the reported number a selection estimate rather than an unbiased one. The grid
is five points wide and the curve is usually flat between the middle three, so the bias is
small — but the honest reading of the table is "this alpha is not worse than the others",
not "fine-tuning bought exactly this much".

The winner is then merged into the base weights. vLLM loads a plain checkpoint far more
reliably than base-plus-adapter for multimodal models, the archive has to carry weights
anyway, and merging removes a moving part from a container we cannot debug interactively.

In [ ]:
FUSION_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]  # weight on the style adapter
fusion_rows = []

if RUN_TRAIN and len(ADAPTERS) == 2:
    _, held_idx = data.holdout_by_conversation(ROWS, EVAL_CONVO)
    held_rows = data.subset(ROWS, held_idx)
    held_refs = [r["text"] for r in held_rows]
    for alpha in FUSION_GRID:
        fused = train_lora.fuse_adapters(
            SPEC,
            adapters=ADAPTERS,
            weights={"lexicon": 1.0 - alpha, "style": alpha},
            out_dir=ARTIFACTS / f"fused-{alpha:.2f}",
        )
        merged = train_lora.merge_and_save(SPEC, fused, ARTIFACTS / f"merged-{alpha:.2f}")
        # After training the GPU still holds leftover blocks; vLLM asking for 0.85 of
        # *total* VRAM then fails. Transformers is enough for a held-out fusion sweep.
        backend = decode.build_backend(merged, prefer="transformers")
        decoder = decode.Qwen3AsrDecoder(backend=backend, length_prior=prior)
        texts = [
            h.text
            for h in decoder.decode_pass(
                held_rows, decode.DecodeConfig(prompt=PASS_SPECS[0][1], name=f"fuse-{alpha}")
            )
        ]
        decoder.close()
        fusion_rows.append(
            {
                "style weight": alpha,
                "held-out WER": round(scoring.corpus_wer(held_refs, texts), 4),
                "+ post-processing": round(
                    scoring.corpus_wer(held_refs, PIPELINE.apply_many(texts, [r["duration"] for r in held_rows])), 4
                ),
            }
        )
    show(pd.DataFrame(fusion_rows).set_index("style weight"))
    best_alpha = min(fusion_rows, key=lambda r: r["+ post-processing"])["style weight"]
    print("selected style weight:", best_alpha)
    train_lora.merge_and_save(
        SPEC, ARTIFACTS / f"fused-{best_alpha:.2f}" / "fused", MERGED_DIR
    )
elif ADAPTERS.get("style") and RUN_TRAIN:
    print("one adapter: merging style adapter directly")
    train_lora.merge_and_save(SPEC, ADAPTERS["style"], MERGED_DIR)
else:
    print("fusion sweep needs both adapters and a GPU; skipped.\n"
          "With one adapter, merge it directly:\n"
          "    train_lora.merge_and_save(SPEC, ADAPTERS['style'], MERGED_DIR)")

if HUB and HUB.token() and (ADAPTERS or MERGED_DIR.exists()):
    print("publishing checkpoints to", HUB.repo_id)
    hub.publish_run(
        HUB,
        adapters={Path(p).name: p for p in ADAPTERS.values()},
        merged_dir=MERGED_DIR if MERGED_DIR.exists() else None,
        tensorboard_dir=RUNS_DIR,
        manifest_path=next(
            (Path(p) / "run_manifest.json" for p in ADAPTERS.values()
             if (Path(p) / "run_manifest.json").exists()),
            None,
        ),
    )
elif HUB and not HUB.token():
    print("LIT_HF_REPO is set but HF_TOKEN is missing; checkpoints stay local")

### 10.3 Publishing the checkpoint

A checkpoint sitting in `artifacts/` on a rented GPU box has a lifetime measured in
billing cycles. Pushing to the Hub costs one cell and buys three things worth having:

* **One name for the artefact.** The notebook, the packaging script and the write-up all
  refer to the same revision string instead of a path that only existed on one machine.
* **Metrics attached to weights.** The event files go up under `runs/`, so the Hub renders
  the loss and held-out-WER curves on the model page next to the checkpoint that produced
  them. A number in a chat log is not evidence; a curve next to the weights is.
* **A reproducibility story that already exists.** Winning solutions have to be verifiable
  and openly licensed. `run_manifest.json` and an MPL-2.0 model card generated from the
  actual run means that requirement is already satisfied at submission time.

Adapters are tens of megabytes and always pushed. The merged 1.7B checkpoint is gigabytes,
so it needs `LIT_HF_UPLOAD_MERGED=1` — worth doing once for the final model, since that is
the exact artefact the archive embeds.

Run this cell again after §13: the ledger and the final `submission_config.json` do not
exist yet on a first pass, and they are the two files that make the repo self-explanatory.

The card is rendered locally, so the preview below works with no token and no network. Set
`HF_TOKEN` and `LIT_HF_REPO` to actually publish.

In [ ]:
PUBLISHED = None
_metrics = [
    {"system": f"{name}/{stage['stage']}", "eval WER": (stage.get("best_wer") or {}).get("eval_wer")}
    for name, history in (curves.items() if have_curves else [])
    for stage in history
    if stage.get("best_wer")
]
_stages = [
    {"name": s.name, "corpus_weights": s.corpus_weights, "epochs": s.epochs,
     "learning_rate": s.learning_rate}
    for s in (S1, S2)
]
_notes = (
    f"Held-out conversation `{EVAL_CONVO}` ({len(EVAL_ROWS)} clips) was excluded from "
    f"training and used for the in-training WER probe and the fusion sweep. Seed {SEED}."
)

if HUB is None:
    print("LIT_HF_REPO not set; nothing will be published. Card preview only.\n")
elif not HUB.token():
    print(f"{HUB.token_env} not set; nothing will be published. Card preview only.\n")
elif not ADAPTERS:
    print("no adapters on disk yet; run §10 on a GPU first.\n")
else:
    _manifest = next(
        (Path(p) / "run_manifest.json" for p in ADAPTERS.values()
         if (Path(p) / "run_manifest.json").exists()),
        None,
    )
    PUBLISHED = hub.publish_run(
        HUB,
        adapters={Path(p).name: p for p in ADAPTERS.values()},
        merged_dir=MERGED_DIR if MERGED_DIR.exists() else None,
        tensorboard_dir=RUNS_DIR,
        manifest_path=_manifest,
        extra_files={
            "submission_config.json": SUBMISSION_SRC / "submission_config.json",
            "postprocess.json": SUBMISSION_SRC / "postprocess.json",
            "ablations.csv": ARTIFACTS / "ablations.csv",
        },
        metrics=_metrics,
        stages=_stages,
        notes=_notes,
    )
    print("\n" + PUBLISHED.url())

_card = hub.render_model_card(
    HUB or hub.HubSpec(repo_id="your-name/lit-jv-qwen3asr-lora"),
    metrics=_metrics,
    manifest={"seed": SEED, "config_fingerprint": repro.config_fingerprint(SPEC),
              "data": repro.manifest_fingerprint(TRAIN_ROWS), "environment": FINGERPRINT},
    stages=_stages,
    notes=_notes,
)
print("\n".join(_card.split("\n")[:34]))
print("...")

Re-run §6 once `artifacts/merged-qwen3asr/` exists — `MODEL_PATH` picks it up
automatically — then re-run §8 and §9 so the rules and the combiner are fitted against the
*fine-tuned* model's error distribution. Rules gated on the zero-shot model's mistakes are
fitted to the wrong system, and this is the step most people skip.

## 11. The ledger

Every measurement made above, in one table. This is what gets copied into the winning-model
documentation, and what stops the final configuration from being chosen by vibes.

In [ ]:
ledger = pd.DataFrame(RESULTS).set_index("system")
show(ledger)
ledger.to_csv(ARTIFACTS / "ablations.csv")

reference = ledger["WER"].iloc[0] if len(ledger) else float("nan")
if len(ledger) > 1:
    show(
        ledger.assign(
            **{"vs first row": (ledger["WER"] - reference).round(4)}
        )[["WER", "vs first row", "note"]]
    )
print(f"\nsaved {len(ledger)} rows to {ARTIFACTS / 'ablations.csv'}"
      + ("  (simulated hypotheses: treat as a wiring check, not a result)" if SIMULATED else ""))

# The ledger is only usable as evidence if it says which run produced it.
# The measured passes are highly correlated.  Ship the best single prompt rather
# than spending the time budget on a vote that was worse on the development set.
SUBMISSION_SPECS = [next(spec for spec in PASS_SPECS if spec[0] == BEST_PASS)]
SUBMISSION_COMBINER = "none"

CONFIG_FOR_HASH = {
    "passes": [(name, cfg.__dict__, temperature) for name, cfg, temperature in SUBMISSION_SPECS],
    "combiner": SUBMISSION_COMBINER,
    "combiner_by_n": {},
    "length_prior": {"mean": prior.mean, "std": prior.std},
    "rules": list(PIPELINE.rules),
    "model": str(MODEL_PATH),
    "seed": SEED,
}
MANIFEST_PATH = repro.write_run_manifest(
    ARTIFACTS / "run_manifest.json",
    seed=SEED,
    deterministic=DETERMINISTIC,
    config_fingerprint=repro.config_fingerprint(CONFIG_FOR_HASH),
    simulated_hypotheses=SIMULATED,
    model_path=str(MODEL_PATH),
    data=repro.manifest_fingerprint(ROWS),
    combiner={"selected": COMBINER, "by_n": COMBINER_BY_N},
    ledger=ledger.reset_index().to_dict("records"),
    artifacts=repro.file_fingerprints(
        [SUBMISSION_SRC / "postprocess.json", ARTIFACTS / "ablations.csv"]
    ),
)


## 12. The time budget is part of the model

Two hours, one A100, and a test set of unknown size — dev is 372 clips for 2.5 h and the
test set is described as larger, so plan for ~1 000 clips. A system that times out scores
nothing, and throughput is the one quantity that cannot be measured until the real machine
runs, so the plan is chosen against a **sweep** of plausible rates rather than a point
estimate, and the choice is made at the pessimistic end.

`main.py` then re-derives the decision at run time, which is the part that actually
protects the submission: it times pass 1 and starts pass *k+1* only if
`1.15 x pass_cost + rescue_reserve` fits in the remaining budget, checkpointing a complete
`submission.csv` after every pass. Being wrong here costs a pass, not the submission.

In [ ]:
TEST_CLIPS = int(os.environ.get("LIT_TEST_CLIPS", 1000))
MEAN_CLIP_S = float(pd.DataFrame(ROWS)["duration"].mean())
LOAD_S = 300.0        # weights off disk + vLLM graph capture, measured on the smoke test
OVERHEAD_S = 120.0    # manifest, csv writes, teardown
SOFT_DEADLINE_S = 6300.0

#: Replace with the clips/s printed by 6 on the target GPU.
MEASURED_CLIPS_PER_S = os.environ.get("LIT_CLIPS_PER_S")
RATE_SWEEP = [0.25, 0.5, 1.0, 2.0]
PLANNING_RATE = float(MEASURED_CLIPS_PER_S) if MEASURED_CLIPS_PER_S else min(RATE_SWEEP[1:])

grid = []
for rate in RATE_SWEEP:
    per_pass = TEST_CLIPS / rate
    row = {"clips/s": rate, "min/pass": round(per_pass / 60)}
    for n in (1, 2, 3, 4):
        total = LOAD_S + n * per_pass + 0.35 * per_pass + OVERHEAD_S
        row[f"{n} pass"] = f"{round(total / 60)}m" + ("" if total < SOFT_DEADLINE_S else " OVER")
    grid.append(row)
show(pd.DataFrame(grid).set_index("clips/s"))

per_pass = TEST_CLIPS / PLANNING_RATE
affordable = [
    n
    for n in range(1, len(PASS_SPECS) + 1)
    if LOAD_S + n * per_pass + 0.35 * per_pass + OVERHEAD_S < SOFT_DEADLINE_S
]
EXPECTED_PASSES = max(affordable) if affordable else 1
print(f"planning rate {PLANNING_RATE} clips/s ({TEST_CLIPS} clips, mean {MEAN_CLIP_S:.0f}s)"
      + ("  [measured]" if MEASURED_CLIPS_PER_S else "  [pessimistic assumption]"))
print(f"at that rate {EXPECTED_PASSES} of {len(PASS_SPECS)} passes complete")
show(pd.DataFrame([decode.estimate_budget(TEST_CLIPS, TEST_CLIPS * MEAN_CLIP_S / 3600,
                                          PLANNING_RATE, load_seconds=LOAD_S)]).T.rename(columns={0: "value"}))

All three passes go into the config anyway. Configuring the ambitious plan is strictly
better than configuring the pessimistic one, because the two guards that make it safe are
already in place: `main.py` refuses to *start* a pass that does not fit, and §9.1 measured
a combiner for every surviving pass count. So a fast GPU gets three passes and a slow one
degrades to two with the right combiner — instead of a fast GPU being capped at two by a
guess made here.

## 13. Packaging the submission

The archive carries everything: entrypoint at the root, the validated `lit_jv` package,
the fitted rules and lexicon, the mined glossary, the length prior, and the merged
checkpoint. No network, no surprises.

`submission_config.json` is the single place where the notebook's findings cross into the
container — passes, combiner, thresholds, prior. Nothing else needs editing to change the
system's behaviour.

In [ ]:
CONFIG = {
    "model_dir": "model",
    "backend": "vllm",
    "time_budget_s": 6300,
    "chunk_size": 48,
    "passes": [
        {
            "name": name,
            "prose": cfg.prose,
            "glossary": bool(cfg.glossary),
            "max_glossary_terms": cfg.max_glossary_terms,
            "force_language": cfg.force_language,
            "temperature": temperature,
        }
        for name, cfg, temperature in SUBMISSION_SPECS
    ],
    "combiner": SUBMISSION_COMBINER,
    "combiner_by_n": {},
    "rover": {"null_penalty": 1.0, "insertion_threshold": 0.5},
    "fallback": {
        "enabled": True,
        "min_avg_logprob": -0.85,
        "max_length_z": 3.0,
        "sampling_draws": 4,
        "sampling_temperature": 0.4,
        "split_long_clips": True,
        "max_rescue_fraction": 0.15,
    },
    "length_prior": {"mean": round(prior.mean, 4), "std": round(prior.std, 4)},
    "postprocess": "postprocess.json",
    "glossary": GLOSSARY,
    "vllm": {"gpu_memory_utilization": 0.85, "max_model_len": 4096},
}
(SUBMISSION_SRC / "submission_config.json").write_text(
    json.dumps(CONFIG, ensure_ascii=False, indent=1), "utf-8"
)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "glossary"}, indent=1))
print(f"+ {len(GLOSSARY)} glossary terms")
print("shipped config fingerprint:", repro.config_fingerprint(CONFIG))

In [ ]:
def run(*command):
    result = subprocess.run(
        [sys.executable, *command], cwd=ROOT, capture_output=True, text=True, encoding="utf-8"
    )
    print((result.stdout or "") + (result.stderr or ""))
    print(f"-> exit {result.returncode}")
    return result.returncode


# 1. Unit tests: scorer parity, prompt parity, combiners, rules.
assert run("tests/test_pipeline.py") == 0
# 2. Full container dry run with a fake decoder: manifest -> passes -> combine -> csv.
assert run("tests/test_submission_dryrun.py") == 0
# 3. A later optional pass must never overwrite a completed checkpoint with blanks.
assert run("tests/test_submission_resilience.py") == 0

In [ ]:
pack = ["scripts/pack_submission.py"]
if MERGED_DIR.exists():
    pack += ["--model", str(MERGED_DIR)]
elif os.environ.get("LIT_HF_CHECKPOINT"):
    # Resolved and embedded now, on a machine with network. The container has none.
    pack += ["--model", os.environ["LIT_HF_CHECKPOINT"],
             "--model-subfolder", os.environ.get("LIT_HF_SUBFOLDER", "merged")]
else:
    print("no merged checkpoint yet: building a CODE-ONLY archive.\n"
          "A real submission must embed the weights - the container has no network.\n")
assert run(*pack) == 0

### Local container test before submitting

The dry run above proves the wiring; only Docker proves the environment. On a Runpod
development pod, first install the pinned Qwen stack and run the CUDA preflight:

```bash
python scripts/install_runpod.py   # pip install + apt-package repair + verify_runpod.py
```

The preflight requires Python 3.12 and verifies Qwen's published compatible versions
of `qwen-asr`, `transformers`, and `vllm`; it does not download a model. Do this only
for the development pod, never inside the submitted archive. From the
[runtime repository](https://github.com/drivendataorg/lost-in-transcription-runtime):

```bash
# build a local stand-in for the container's read-only data/ directory
python scripts/make_local_test_data.py --out runtime/data
cp submission/submission.zip runtime/submission/submission.zip

make test-submission            # runs the real image against runtime/data

python score.py runtime/ground_truth.csv \
    --predicted-path runtime/submission/submission.csv
```

Use `--limit 8` when building the local data to rehearse the smoke-test environment,
which allows one minute in total — enough to expose a model-load time that will not fit.

Then submit as a **smoke test** first (1-minute limit, does not count against the
submission cap), and only then as a full submission.

## 14. Submission checklist

- [ ] `main.py` at the **root** of the zip, not inside a folder.
- [ ] Merged weights under `model/`; no Hub ids left in `submission_config.json`.
- [ ] `scoring.self_check()` passes with the shipped `lit_jv`.
- [ ] Post-processing rules were re-fitted against the *final* model's hypotheses (§8 after §10).
- [ ] Held-out-conversation delta for the rule set is negative; the fold table is saved.
- [ ] `combiner_by_n` re-measured on real hypotheses (§9.1), including the "none" option.
- [ ] `time_budget_s` <= 6300, and §12 says at least one pass completes at the pessimistic rate.
- [ ] Placeholder `submission.csv` is written before the model loads (verified by the dry run).
- [ ] No log line contains a filename, a transcript, or any test-set content.
- [ ] Smoke test green before the full submission.
- [ ] `run_manifest.json` sits next to the merged weights, and its commit is not dirty.
- [ ] Final model retrained on **all** dev conversations after the held-out probe was read.
- [ ] Checkpoint, event files and manifest pushed to the Hub, repo made public.

## 15. What the next hour of work should buy

Ordered by expected WER per hour spent, given the error budget in §3:

1. **Re-fit §8 on fine-tuned hypotheses.** Free; the rules currently price the zero-shot
   model's mistakes. Usually the single largest remaining gain.
2. **More Javanese audio from the MDC catalog** into stage 1 (`load_flat_corpus` already
   ingests the flat `metadata.tsv` + `clips/` layout used across the catalog). Lexical
   coverage is the biggest non-style error class in the taxonomy.
3. **Widen the fusion sweep** around the selected alpha, and add a third adapter trained on
   Indonesian-only turns — the 5% monolingual slice has its own convention.
4. **A fourth, genuinely decorrelated pass** — for example a Whisper-large-v3 pass whose
   output is style-repaired and voted in — but only if §12 says it fits. The oracle row in
   §9 tells you the ceiling before you spend the time.
5. **Sharpen the length prior per speaker.** Speaking rate varies enough between the four
   dev speakers that a single global sigma over-flags the fast ones.

Everything above is measured, seeded, fingerprinted, reproducible from `artifacts/`, and gated on a held-out
conversation. That combination — not any single trick — is what makes a leaderboard number
hold up on a private split.